# Contextual Compression Index

**1. 검색 속도 및 인덱싱 효율성 향상**

* 청크 수의 감소: 원문을 잘게 쪼개어 수만 개의 청크로 만드는 대신, 문서당 하나의 요약문만 인덱싱한다면 전체 벡터의 개수($N$)가 줄어든다. 결과적으로 전체 인덱스 용량($N \times d$)이 줄어들어 메모리를 절약하게 된다.

* **메모리(RAM) 절약:** 인덱스 크기가 작아지므로 동일한 메모리 사양에서 더 많은 문서를 관리하거나, 더 저렴한 인프라 비용으로 시스템을 운영할 수 있다.

**2. 검색 노이즈 제거 (Denoising)**

* **의미적 응집도 강화:** 문서에는 질문과 상관없는 부차적인 설명이나 수식어구가 포함되어 있다. 인덱싱 단계에서 이를 제거하고 핵심 의미(Semantic core)만 남겨 저장하면, 검색 시 질문 벡터와 문서 벡터 간의 정합성이 높아져 검색 정확도가 향상된다.
* **장문 문서 처리:** 아주 긴 문서는 임베딩 모델의 토큰 제한에 걸려 정보가 소실될 수 있다. 이를 미리 요약하여 압축 인덱스에 넣으면 문서 전체의 맥락을 검색 단계에서 더 잘 반영할 수 있다.

**3. 'Multi-Vector' 전략의 활용**

* **요약문 기반 검색, 원문 기반 답변:** 압축 인덱스(`ir-compressed`)에서 질문과 가장 관련 있는 요약본을 빠르게 찾아낸 뒤, 실제 LLM에게 전달할 때는 해당 요약본과 연결된(Parent-Child 관계) 원문(`ir`)을 불러오는 구조를 취할 수 있다.
* 이를 통해 **"검색은 가볍고 정확하게, 답변은 풍부한 맥락으로"** 수행하는 하이브리드 접근이 가능하다.

**4. 검색 품질의 일관성 유지**

* 다양한 포맷(PDF, HTML, 로그 파일 등)의 데이터를 표준화된 요약 형태로 인덱싱하면, 데이터 소스의 형태에 구애받지 않고 일관된 검색 성능을 기대할 수 있다.

**요약 및 비교**

| 항목 | 원문 인덱스 검색 | 압축 인덱스 검색 (ir-compressed) |
| --- | --- | --- |
| **검색 대상** | 문서 청크 전체 | 요약문, 키워드, 혹은 핵심 추출문 |
| **주요 장점** | 정보 손실 없음 | 검색 속도 빠름, 노이즈 적음 |
| **데이터 구조** | 단순 저장 | Parent-Child 또는 요약-원문 매핑 필요 |

이러한 방식은 특히 **데이터 양이 방대하여 검색 비용이 부담되거나, 문서 하나하나의 길이가 길어 검색 정확도가 떨어질 때** 매우 유용하다.

In [ ]:
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

# Pinecone 인덱스와 동일한 1536차원 임베딩 모델
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small'
)

print('환경변수와 임베딩 모델 로드 완료!')

In [ ]:
# 원본 문서와 평가용 질의 데이터 불러오기
document_df = pd.read_csv('./documents.csv')
queries_df = pd.read_csv('./queries.csv')

display(document_df.head())
display(queries_df.head())

### 문서 요약 체인 및 문서 압축

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from tqdm.auto import tqdm
import time

llm = init_chat_model('gpt-5.6-luna', temperature=0.3)

prompt = PromptTemplate.from_template('''
당신은 정보의 본질을 빠르고 정확하게 파악하는 요약 전문가입니다.
다음 지문에서 핵심 내용만 효과적이고 효율적으로 요약해 주세요.

{text}
''')

output_parser = StrOutputParser()
summary_chain = prompt | llm | output_parser
compressed_texts = []

for idx, row in tqdm(
    document_df.iterrows(),
    total=len(document_df),
    desc='문서 요약/압축',
    ncols=150
):
    doc_id = row['doc_id']
    content = row['content']
    summary = summary_chain.invoke({'text': content})
    compressed_texts.append({'doc_id': doc_id, 'content': summary})
    time.sleep(0.5)

compressed_df = pd.DataFrame(compressed_texts)
display(compressed_df)

In [ ]:
# 원문과 압축문을 병합하여 비교
pd.set_option('display.max_colwidth', None)

comparison_df = pd.merge(
    document_df[['doc_id', 'content']],
    compressed_df,
    on='doc_id',
    suffixes=('_original', '_compressed')
)

display(comparison_df)

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()
index_names = pc.list_indexes().names()
print(index_names)

# 압축 문서를 저장할 별도 인덱스 생성
if 'ir-compressed' not in index_names:
    pc.create_index(
        name='ir-compressed',
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(region='us-east-1', cloud='aws')
    )
    print('ir-compressed 인덱스 생성 완료!')
else:
    print('ir-compressed 인덱스가 이미 존재합니다.')

In [ ]:
from langchain_pinecone import PineconeVectorStore

# 압축 문서 인덱스
vector_store = PineconeVectorStore(
    index_name='ir-compressed',
    embedding=embeddings
)

# 원본 문서 인덱스
original_vector_store = PineconeVectorStore(
    index_name='ir',
    embedding=embeddings
)

print('원본 및 압축 벡터스토어 연결 완료!')

In [ ]:
from langchain_core.documents import Document
from pprint import pprint

# 압축 DataFrame을 LangChain Document로 변환
docs_to_index = []

for idx, row in tqdm(
    compressed_df.iterrows(),
    total=len(compressed_df)
):
    doc_id = row['doc_id']
    content = row['content']
    doc = Document(
        page_content=content,
        metadata={'doc_id': doc_id}
    )
    docs_to_index.append(doc)

pprint(len(docs_to_index))

# 동일한 ID를 사용해 재실행 시 기존 레코드 갱신
vector_store.add_documents(
    documents=docs_to_index,
    ids=[doc.metadata['doc_id'] for doc in docs_to_index]
)

print('압축 문서 업로드 완료!')

In [ ]:
# 검색 성능 평가 함수
def parse_relevant(relevant_str) -> dict[str, int]:
    relevant_dict = {}
    for pair in relevant_str.split(';'):
        doc_id, grade = pair.split('=')
        relevant_dict[doc_id] = int(grade)
    return relevant_dict


def compute_metrics(predicted, relevant_dict, k=5):
    hits = sum(1 for doc_id in predicted[:k] if doc_id in relevant_dict)
    precision = hits / k

    total_relevant = len(relevant_dict)
    recall = hits / total_relevant if total_relevant > 0 else 0

    rr = 0
    for rank, doc_id in enumerate(predicted, 1):
        if doc_id in relevant_dict:
            rr = 1 / rank
            break

    num_correct = 0
    precisions = []
    for rank, doc_id in enumerate(predicted[:k], 1):
        if doc_id in relevant_dict:
            num_correct += 1
            precisions.append(num_correct / rank)

    ap = np.mean(precisions) if precisions else 0
    return precision, recall, rr, ap


def evaluate_all(method_results, queries_df, k=5):
    precision_list, recall_list, rr_list, ap_list = [], [], [], []

    for _, row in queries_df.iterrows():
        query_id = row['query_id']
        relevant_dict = parse_relevant(row['relevant_doc_ids'])
        predicted = method_results[query_id]
        precision, recall, rr, ap = compute_metrics(predicted, relevant_dict, k)
        precision_list.append(precision)
        recall_list.append(recall)
        rr_list.append(rr)
        ap_list.append(ap)

    return {
        'P@k': np.mean(precision_list),
        'R@k': np.mean(recall_list),
        'MRR': np.mean(rr_list),
        'MAP': np.mean(ap_list)
    }

In [ ]:
original_results = {}
compressed_results = {}
top_k = 5

for idx, row in tqdm(
    queries_df.iterrows(),
    total=len(queries_df),
    desc='원본/압축 검색 평가'
):
    qid = row['query_id']
    query_text = row['query_text']

    docs_from_original = original_vector_store.similarity_search(query_text, k=top_k)
    original_results[qid] = [
        doc.metadata['doc_id'] for doc in docs_from_original
    ]

    docs_from_compressed = vector_store.similarity_search(query_text, k=top_k)
    compressed_results[qid] = [
        doc.metadata['doc_id'] for doc in docs_from_compressed
    ]

print('검색 결과 생성 완료!')

In [ ]:
original_metrics = evaluate_all(
    original_results,
    queries_df,
    k=top_k
)

compressed_metrics = evaluate_all(
    compressed_results,
    queries_df,
    k=top_k
)

print('Original:', original_metrics)
print('Compressed:', compressed_metrics)

In [ ]:
metrics_df = pd.DataFrame({
    'Metrics': [f'P@{top_k}', f'R@{top_k}', 'MRR', 'MAP'],
    'Original': [
        original_metrics['P@k'],
        original_metrics['R@k'],
        original_metrics['MRR'],
        original_metrics['MAP']
    ],
    'Compressed': [
        compressed_metrics['P@k'],
        compressed_metrics['R@k'],
        compressed_metrics['MRR'],
        compressed_metrics['MAP']
    ]
})

display(metrics_df)

In [ ]:
import matplotlib.pyplot as plt

metrics = metrics_df['Metrics']
original_vals = metrics_df['Original']
compressed_vals = metrics_df['Compressed']
x = np.arange(len(metrics))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 막대그래프
axes[0].bar(x - width / 2, original_vals, width, label='Original')
axes[0].bar(x + width / 2, compressed_vals, width, label='Compressed')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Original vs Compressed - Bar Chart')
axes[0].set_ylabel('Score')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# 선그래프
axes[1].plot(metrics, original_vals, marker='o', linewidth=2, label='Original')
axes[1].plot(metrics, compressed_vals, marker='s', linewidth=2, label='Compressed')
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Original vs Compressed - Line Chart')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()